# Task 2 — Data Discovery, Profiling and Cleaning

## Objective

This notebook loads the raw Pluralsight article data generated during Task 1.

The data is profiled to identify:

- Number of rows and columns
- Column names
- Data types
- Missing values
- Unique values
- Duplicate records
- Mixed data types
- Nested fields
- Other data quality issues

The identified issues are then addressed through cleaning and standardization.

The final cleaned dataset is saved as:

`../data/interim/cleaned.csv`

## Imports

In [13]:
import json
import os
import re
import pandas as pd

## Data Ingestion

In [14]:
# ==============================================================================
# INPUT / OUTPUT DECLARATIONS 
# ==============================================================================
AI_DATA_INPUT_PATH = "../data/raw/pluralsight_ai_data_articles.json"
CLOUD_INPUT_PATH = "../data/raw/pluralsight_cloud_articles.json"

INTERIM_DIR = "../data/interim"
OUTPUT_CSV_PATH = os.path.join(INTERIM_DIR, "cleaned.csv")

# Safely verify the directory exists without overwriting or destroying your files
os.makedirs(INTERIM_DIR, exist_ok=True)

# Load JSON lists cleanly into memory DataFrame arrays
raw_ai_records = []
if os.path.exists(AI_DATA_INPUT_PATH):
    with open(AI_DATA_INPUT_PATH, "r", encoding="utf-8") as f:
        raw_ai_records = json.load(f)
else:
    print(f"Warning: Could not find existing file at {AI_DATA_INPUT_PATH}")

raw_cloud_records = []
if os.path.exists(CLOUD_INPUT_PATH):
    with open(CLOUD_INPUT_PATH, "r", encoding="utf-8") as f:
        raw_cloud_records = json.load(f)
else:
    print(f"Warning: Could not find existing file at {CLOUD_INPUT_PATH}")

# Convert into working DataFrames
df_ai = pd.DataFrame(raw_ai_records)
df_cloud = pd.DataFrame(raw_cloud_records)

# Concatenate both subsets into a unified operational collection space
df_raw = pd.concat([df_ai, df_cloud], ignore_index=True)
print(f"Loaded a total unified dataset size of: {df_raw.shape[0]} rows across {df_raw.shape[1]} columns.")


Loaded a total unified dataset size of: 20 rows across 10 columns.


## Data Discovery & Profiling Summary Table

In [15]:
# Create the profiling metric array explicitly required by project deliverable constraints
profiling_metrics = []

for column in df_raw.columns:
    col_data = df_raw[column]

    # Analyze core properties
    dtype_str = str(col_data.dtype)
    null_count = int(col_data.isnull().sum())
    percent_null = float((null_count / len(df_raw)) * 100)

    # Track unique items accurately (convert lists to strings briefly to prevent unhashable object type exceptions)
    hashable_series = col_data.apply(
        lambda x: str(x) if isinstance(x, list) else x
    )
    unique_count = int(hashable_series.nunique())

    # Grab a descriptive valid sample item if present
    sample_val = (
        col_data.dropna().iloc[0] if len(col_data.dropna()) > 0 else "N/A"
    )

    profiling_metrics.append({
        "column_name": column,
        "data_type": dtype_str,
        "null_count": null_count,
        "percent_null_ratio": f"{percent_null:.2f}%",
        "unique_count": unique_count,
        "sample_value_preview": str(sample_val)[:50],
    })

df_profile_summary = pd.DataFrame(profiling_metrics)

print("=========================================================================")
print("TASK 2 - DATA PROFILING SUMMARY SUMMARY")
print("=========================================================================")
display(df_profile_summary)


TASK 2 - DATA PROFILING SUMMARY SUMMARY


,column_name,data_type,null_count,percent_null_ratio,unique_count,sample_value_preview
0,source,str,0,0.00%,1,Pluralsight
1,category,str,0,0.00%,2,AI & Data
2,title,str,0,0.00%,20,10 emerging AI jobs to watch
3,author,str,0,0.00%,8,Steve Buchanan
4,publication_date,str,0,0.00%,19,"Jun 15, 2026"
5,description,str,0,0.00%,20,"Discover ten high-paying, emerging AI jobs tha..."
6,tags,object,0,0.00%,16,"['Upskilling', 'Business & Leadership', 'AI & ..."
7,url,str,0,0.00%,20,https://pluralsight.com/resources/blog/ai-and-...
8,content,str,0,0.00%,20,10 emerging AI jobs to watch\n\n\nDiscover ten...
9,scraped_at,str,0,0.00%,20,2026-09-12T18:44:43.864602


## Documented Quality Issues Tracker

In [16]:
# To satisfy the mandated 'Issues list' report, we perform programmatic assertions over structural gaps
print("=========================================================================")
print("DATA QUALITY ANOMALY LOGS")
print("=========================================================================")

# 1. Look for row duplicates matching identical URL structures
url_dupes = df_raw.duplicated(subset=["url"]).sum() if "url" in df_raw.columns else 0
print(f" Anomaly Issue 1 [Duplicates]: Found {url_dupes} duplicate web paths across active row arrays.")

# 2. Track down blank authors or missing title entities
blank_authors = df_raw[df_raw["author"] == ""].shape[0] if "author" in df_raw.columns else 0
print(f" Anomaly Issue 2 [Missing Metadata]: Found {blank_authors} records with empty metadata author bylines.")

# 3. Discover nested tag objects that violate relational design standard shapes
has_nested_lists = df_raw["tags"].apply(lambda x: isinstance(x, list)).any() if "tags" in df_raw.columns else False
print(f" Anomaly Issue 3 [Nested Structs]: Nested python list objects located inside 'tags' columns. Action Required: Flatten columns.")


DATA QUALITY ANOMALY LOGS
 Anomaly Issue 1 [Duplicates]: Found 0 duplicate web paths across active row arrays.
 Anomaly Issue 2 [Missing Metadata]: Found 0 records with empty metadata author bylines.
 Anomaly Issue 3 [Nested Structs]: Nested python list objects located inside 'tags' columns. Action Required: Flatten columns.


## Normalization, Parsing, & Flattening Execution

In [17]:
def to_snake_case(name):
    """Transforms tracking identifiers explicitly to snake_case rules."""
    s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
    return re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1).lower().strip()


# Create a clean operational mutation workflow pointer space
df_clean = df_raw.copy()

# 1. Flatten the lists of tags to comma-separated strings so they survive standard CSV structural exports
if "tags" in df_clean.columns:
    df_clean["tags"] = df_clean["tags"].apply(
        lambda x: ", ".join(x) if isinstance(x, list) else str(x)
    )

# 2. STANDARDIZE PUBLICATION DATES (Convert Text Strings into Proper YYYY-MM-DD dates)
if "publication_date" in df_clean.columns:
    # errors='coerce' turns unparseable dates into NaT (Not a Time) instead of crashing the pipeline
    df_clean["publication_date"] = pd.to_datetime(
        df_clean["publication_date"], errors="coerce"
    ).dt.strftime("%Y-%m-%d")

# 3. De-duplicate actual tracking row frames on disk explicitly
if "url" in df_clean.columns:
    df_clean = df_clean.drop_duplicates(subset=["url"])

# 4. Clean string column whitespace bounds comprehensively
for col in df_clean.select_dtypes(include=["object"]).columns:
    if col != "tags" and col != "publication_date":  # Avoid re-splitting processed formats
        df_clean[col] = df_clean[col].astype(str).str.strip()

# 5. Enforce clean column naming criteria
df_clean.columns = [to_snake_case(col) for col in df_clean.columns]

print(
    f"Data mutation transformations finished. Working tracking shape is down to: {df_clean.shape} uniform rows."
)


Data mutation transformations finished. Working tracking shape is down to: (20, 10) uniform rows.


C:\Users\Sarah\AppData\Local\Temp\ipykernel_32380\3644180994.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_clean.select_dtypes(include=["object"]).columns:


## Checkpoint Target Export & Output Verification

In [18]:
# Write the finalized processed interim records state directly to disk as requested in rules
df_clean.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8")
print(f"File flushed smoothly into relative environment storage destination path: {OUTPUT_CSV_PATH}\n")

# Verify disk integrity by performing immediate local reading assertions
df_verification = pd.read_csv(OUTPUT_CSV_PATH)
print("=========================================================================")
print("INTERIM CHECKPOINT DISK VALIDATION DETAILS")
print("=========================================================================")
print(f"CSV Rows Read:    {df_verification.shape[0]}")
print(f"CSV Column Names: {list(df_verification.columns)}")
print("\nFirst row sneak-preview column properties:")
print(df_verification.iloc[0].to_dict())


File flushed smoothly into relative environment storage destination path: ../data/interim\cleaned.csv

INTERIM CHECKPOINT DISK VALIDATION DETAILS
CSV Rows Read:    20
CSV Column Names: ['source', 'category', 'title', 'author', 'publication_date', 'description', 'tags', 'url', 'content', 'scraped_at']

First row sneak-preview column properties:
{'source': 'Pluralsight', 'category': 'AI & Data', 'title': '10 emerging AI jobs to watch', 'author': 'Steve Buchanan', 'publication_date': '2026-06-15', 'description': 'Discover ten high-paying, emerging AI jobs that help organizations build, govern, secure, and scale AI.', 'tags': 'Upskilling, Business & Leadership, AI & Data', 'url': 'https://pluralsight.com/resources/blog/ai-and-data/10-emerging-ai-jobs', 'content': "10 emerging AI jobs to watch\n\n\nDiscover ten high-paying, emerging AI jobs that help organizations build, govern, secure, and scale AI.\n\nMention artificial intelligence (AI) and most people still picture traditional data scie